# 05｜DPO 偏好训练

130U-FinLLM｜130U 金融大语言模型

从选定的 SFT 检查点建立策略和冻结参考模型，使用训练分区的已审核偏好对。chosen 与 rejected 各只有一个标签，且两者不同。DPO 的效果不保证优于 SFT；在开发集上比较后再决定最终模型。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
command(sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", "tokenizers>=0.21,<0.23", "scikit-learn>=1.5,<2")
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps")

def finllm(*args):
    command(sys.executable, "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
# 可选持久化：需要你在 Google 的授权界面完成登录。
MOUNT_DRIVE = False  # 需要备份 / 恢复时设为 True，再执行此单元格。
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive 项目目录：", DRIVE_ROOT)
else:
    print("尚未挂载 Drive；/content 是临时磁盘。正式训练前请启用备份。")

In [ ]:
# 新 Notebook 可能分配新的虚拟机；按需恢复前一阶段的文件。
RESTORE_FROM_DRIVE = False
LIVE_STAGES_TO_RESTORE = []  # 中断恢复时填写，例如 ["pretrain"]；会覆盖本地对应阶段。
if RESTORE_FROM_DRIVE:
    if not MOUNT_DRIVE:
        raise RuntimeError("请先把 MOUNT_DRIVE 设为 True 并完成授权。")
    for folder in ("data", "tokenizer", "runs"):
        source = DRIVE_ROOT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", PROJECT / folder, "--working-copy")
    for stage in LIVE_STAGES_TO_RESTORE:
        if stage not in ("pretrain", "domain", "sft", "dpo"):
            raise ValueError("未知检查点阶段：" + str(stage))
        source = DRIVE_ROOT / "live_checkpoints" / stage
        if not source.exists():
            raise FileNotFoundError("缺少阶段自动备份：" + str(source))
        finllm("sync", "--source", source, "--destination", PROJECT / "runs" / stage, "--working-copy")
    print("恢复完成。自动检查点只用于匹配原配置、tokenizer 和数据的继续训练；请指定 RESUME_CHECKPOINT。")

In [ ]:
CONFIG = "configs/dpo.json"
RESUME_CHECKPOINT = None  # 中断恢复或试跑转正式时，填写 "runs/dpo/latest.pt"。
require(CONFIG, 'runs/sft/best.pt', 'tokenizer/english/tokenizer.json', 'data/task/preferences_train.jsonl', 'data/task/development.jsonl')
print(Path(CONFIG).read_text(encoding="utf-8"))

In [ ]:
# 正式训练前持久化输入，避免运行时中断后丢失 tokenizer 和数据。
if not MOUNT_DRIVE:
    raise RuntimeError("请先启用 MOUNT_DRIVE 并完成授权，再备份正式训练输入。")
for folder in ("data", "tokenizer"):
    finllm("sync", "--source", PROJECT / folder, "--destination", DRIVE_ROOT / folder)
shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
print("训练输入已备份；训练中会另外自动同步完整检查点。")

In [ ]:
RUN_TRAINING = False  # 阅读配置、检查 GPU 与数据后设为 True。
if not RUN_TRAINING:
    raise RuntimeError("尚未启动正式训练。确认当前阶段与预算后，把 RUN_TRAINING 设为 True。")
if not torch.cuda.is_available():
    raise RuntimeError("正式 24M 训练需要 GPU；请切换到 T4 运行时。")
if not MOUNT_DRIVE:
    raise RuntimeError("正式训练使用 Drive 自动检查点备份，请先完成挂载。")
runtime_config = json.loads(Path(CONFIG).read_text(encoding="utf-8"))
stage = runtime_config["stage"]
runtime_config["backup_dir"] = str(DRIVE_ROOT / "live_checkpoints" / stage)
runtime_path = Path("configs/runtime") / (stage + "_colab.json")
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(runtime_config, indent=2) + "\n", encoding="utf-8")
arguments = ["train", "--config", str(runtime_path)]
if RESUME_CHECKPOINT:
    require(RESUME_CHECKPOINT, str(RESUME_CHECKPOINT) + ".json")
    arguments += ["--resume", RESUME_CHECKPOINT]
finllm(*arguments)

In [ ]:
if MOUNT_DRIVE:
    for folder in ("data", "tokenizer", "runs"):
        source = PROJECT / folder
        if source.exists():
            finllm("sync", "--source", source, "--destination", DRIVE_ROOT / folder)
    shutil.copy2(PROJECT / "runtime_revision.txt", DRIVE_ROOT / "runtime_revision.txt")
    print("备份完成。请保留校验清单，下一运行时按清单恢复。")
else:
    print("成果当前仅在临时运行时中；请挂载 Drive 后重新运行备份单元格。")